In [ ]:
predictor_variables = [col for col in subset.columns if col not in ['10cm_area', '10cm_volume', '30cm_area', '30cm_volume']]

# Define X (predictors) and Y (response)
X = subset[predictor_variables]
y = subset['10cm_area']

In [ ]:
rf = RandomForestRegressor(n_estimators=100,max_depth=8,random_state=42)
lr = LinearRegression()
cv_parameters = KFold(n_splits=5, shuffle=True, random_state=42)
error_metrics = ['neg_mean_absolute_error', 'r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_percentage_error']

In [ ]:
# Fit once on everything, same as before
y_pred_lr = cross_val_predict(lr, X, y, cv=cv, n_jobs=-1)
y_pred_rf = cross_val_predict(rf, X, y, cv=cv, n_jobs=-1)

# Attach predictions + period back onto the dataframe so you can slice by period
results = subset.copy()
results['y_true'] = y.values
results['pred_lr'] = y_pred_lr
results['pred_rf'] = y_pred_rf

periods = {'early (1980-2013)': (1980, 2013),'mid (2014-2046)':   (2014, 2046), 'late (2047-2080)':  (2047, 2080)}

fig, axes = plt.subplots(2, len(periods), figsize=(5.5*len(periods), 10), sharex=True, sharey=True)

for col, (label, (y0, y1)) in enumerate(periods.items()):
    print(label)
    mask = (results['start_year'] >= y0) & (results['start_year'] <= y1)
    sub = results.loc[mask]

    for row, (pred_col, name) in enumerate(zip(['pred_lr', 'pred_rf'], ['Linear regression', 'Random forest'])):
        ax = axes[row, col]
        r2 = r2_from_pred(sub['y_true'], sub[pred_col])
        rmse = rmse_from_pred(sub['y_true'], sub[pred_col])

        ax.scatter(sub['y_true'], sub[pred_col], alpha=0.3, s=15, edgecolor='none')
        lims = [min(sub['y_true'].min(), sub[pred_col].min()), max(sub['y_true'].max(), sub[pred_col].max())]
        ax.plot(lims, lims, 'k--', linewidth=1)
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_aspect('equal')
        ax.set_title(f'{name} — {label}\nR² = {r2:.3f}, RMSE = {rmse:.3f}, n = {mask.sum()}')
        if col == 0:
            ax.set_ylabel('Predicted')
        if row == 1:
            ax.set_xlabel('Observed')

plt.tight_layout()
plt.show()